# 41 — Stochastic Resonance Under FIM

**UNPRECEDENTED:** Stochastic resonance (SR) — where noise IMPROVES
signal detection — is well-studied in bistable systems. In standard
Kuramoto, noise generally hurts sync (NB27 confirmed monotonic degradation).

But NB27 tested at K=12 (near-threshold). At WEAK coupling (K≈2-5),
where the system is sub-critical, noise might help by:
1. Kicking oscillators out of local phase traps → exploring more of phase space
2. Activating the FIM sensitivity term (near R≈0, sensitivity≈1; fluctuations
   to R>0 trigger FIM feedback that pulls phases together)

This would be a NEW type of SR: **self-observation-mediated stochastic resonance**.

In [ ]:
import numpy as np
import json
from pathlib import Path

import sys
sys.path.insert(0, '/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src')
from scpn_quantum_control.bridge.knm_hamiltonian import build_knm_paper27, OMEGA_N_16

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity

def simulate_R(K_scale, lam, noise, dt=0.02, T=150.0, n_seeds=5):
    K = K_base * K_scale
    Rs = []
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        theta = rng.uniform(0, 2*np.pi, N)
        n_steps = int(T / dt)
        R_tail = []
        for s in range(n_steps):
            diff = theta[None, :] - theta[:, None]
            coupling = np.sum(K * np.sin(diff), axis=1) / N
            dphi = omega + coupling
            if lam > 0:
                dphi += lam * fim_gradient_all(theta)
            theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2*np.pi)
            if s >= n_steps * 3 // 4:
                R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
        Rs.append(np.mean(R_tail))
    return float(np.mean(Rs)), float(np.std(Rs))

print('Ready.')

In [ ]:
# Sweep noise at different (K, λ) regimes
noise_levels = [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]

configs = [
    (2, 0.0, 'weak K, no FIM'),
    (2, 1.0, 'weak K, FIM λ=1'),
    (2, 3.0, 'weak K, FIM λ=3'),
    (5, 0.0, 'moderate K, no FIM'),
    (5, 1.0, 'moderate K, FIM λ=1'),
    (5, 3.0, 'moderate K, FIM λ=3'),
]

sr_results = {}
for K_scale, lam, label in configs:
    print(f'\n=== {label} ===')
    R_vs_noise = []
    for noise in noise_levels:
        R, R_std = simulate_R(K_scale, lam, noise)
        R_vs_noise.append(R)
        print(f'  σ={noise:5.2f}: R={R:.4f} ± {R_std:.4f}')
    
    sr_results[label] = R_vs_noise
    
    # Detect SR: is there a noise level where R is HIGHER than at noise=0?
    R_zero = R_vs_noise[0]
    peak_idx = np.argmax(R_vs_noise)
    R_peak = R_vs_noise[peak_idx]
    
    if R_peak > R_zero + 0.02 and peak_idx > 0:
        print(f'  *** STOCHASTIC RESONANCE DETECTED ***')
        print(f'  Optimal noise: σ={noise_levels[peak_idx]:.2f}, R={R_peak:.4f} vs R(σ=0)={R_zero:.4f}')
        print(f'  Improvement: +{R_peak - R_zero:.4f} ({(R_peak-R_zero)/R_zero*100:.1f}%)')
    else:
        print(f'  No SR: noise monotonically degrades sync.')

In [ ]:
# Summary
print('\n=== STOCHASTIC RESONANCE SUMMARY ===')
for label, R_vs_noise in sr_results.items():
    R_zero = R_vs_noise[0]
    peak_idx = np.argmax(R_vs_noise)
    R_peak = R_vs_noise[peak_idx]
    has_sr = R_peak > R_zero + 0.02 and peak_idx > 0
    print(f'{label:>30}: R(0)={R_zero:.3f}, R_peak={R_peak:.3f} at σ={noise_levels[peak_idx]:.2f}'
          f'  {"SR!" if has_sr else "no SR"}')

# Save
output = {
    'experiment': 'Stochastic resonance under FIM',
    'N': N, 'noise_levels': noise_levels,
    'results': {k: [round(r, 4) for r in v] for k, v in sr_results.items()},
}
out_path = Path('/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/stochastic_resonance_2026-03-29.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nResults saved to {out_path}')